In [1]:
!pip install pyspark --quiet

In [2]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
!wget -q https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

## Q1 - spark version

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('hw6') \
    .getOrCreate()

spark.version

'4.0.2'

## Q2 - read data, repartition and check file sizes

In [4]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
df.repartition(4).write.mode('overwrite').parquet('output/yellow_nov_2025')

In [6]:
import os

files = [f for f in os.listdir('output/yellow_nov_2025') if f.endswith('.parquet')]
sizes = [os.path.getsize(f'output/yellow_nov_2025/{f}') / (1024*1024) for f in files]

for f, s in zip(files, sizes):
    print(f, round(s, 2), 'MB')

print('avg:', round(sum(sizes)/len(sizes), 2), 'MB')

part-00003-ae59394a-c727-4605-b99a-25eef77a6af0-c000.snappy.parquet 25.34 MB
part-00000-ae59394a-c727-4605-b99a-25eef77a6af0-c000.snappy.parquet 25.35 MB
part-00001-ae59394a-c727-4605-b99a-25eef77a6af0-c000.snappy.parquet 25.33 MB
part-00002-ae59394a-c727-4605-b99a-25eef77a6af0-c000.snappy.parquet 25.33 MB
avg: 25.34 MB


## Q3 - trips on nov 15

In [7]:
from pyspark.sql.functions import col, to_date

df.filter(to_date(col('tpep_pickup_datetime')) == '2025-11-15').count()

162604

## Q4 - longest trip

In [8]:
from pyspark.sql.functions import unix_timestamp, max as spark_max

df.withColumn(
    'duration_hours',
    (unix_timestamp('tpep_dropoff_datetime') - unix_timestamp('tpep_pickup_datetime')) / 3600
).agg(spark_max('duration_hours')).show()

+-------------------+
|max(duration_hours)|
+-------------------+
|  90.64666666666666|
+-------------------+



## Q5 - spark ui port

port 4040, http://localhost:4040

## Q6 - least frequent pickup zone

In [9]:
zones = spark.read.option('header', 'true').csv('taxi_zone_lookup.csv')

df.join(zones, df.PULocationID == zones.LocationID) \
  .groupBy('Zone') \
  .count() \
  .orderBy('count') \
  .show(5, truncate=False)

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
+---------------------------------------------+-----+
only showing top 5 rows
